# 08 — Differential DNA methylation

Does the insulin-resistance label the model assigned in notebook 07 leave a trace
in DNA methylation? The 1,199 Taiwan Biobank participants with array data are
split by that *predicted* label — 953 IR− against 246 IR+ — and every autosomal
CpG probe is tested for a difference between the two groups.

**What is being tested, and what that means.** The grouping variable is a model
prediction, not a measurement. A probe that separates the groups is a probe that
tracks the model's notion of insulin resistance — which is itself a function of
the clinical variables — so this is a hypothesis-generating analysis, not
evidence of a methylation–insulin relationship independent of those variables.

**Where this notebook starts.** It reads per-participant methylation tables
rather than the vendor array archive those tables were extracted from. That
archive is restricted-access Taiwan Biobank data and is not distributed with this
repository; `src/methylation/extract.py` documents the extraction step for
researchers who have their own approved access.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import polars as pl

from src.data.io import output_path, processed_path, write_parquet
from src.logging_utils import configure_logging
from src.methylation.annotate import classify_coding_region, load_probe_annotation
from src.methylation.differential import (
    LOG2FC_THRESHOLD,
    P_VALUE_THRESHOLD,
    adjust_pvalues,
    probe_statistics,
    volcano_frame,
)
from src.methylation.extract import annotation_path, assemble_beta_matrix, autosomal_cg_probes
from src.viz.figures import plot_volcano
from src.viz.palettes import (
    COLOR_BLIND_CATEGORICAL,
    NEUTRAL_GREY,
    OKABE_ITO,
)

configure_logging(ROOT / "logs")

twb = pl.read_parquet(processed_path("TWB_with_IR.parquet")).select(["MET_ID", "IR"])
twb = twb.filter(pl.col("MET_ID") != "")

negative_ids = twb.filter(pl.col("IR") == 0)["MET_ID"].to_list()
positive_ids = twb.filter(pl.col("IR") == 1)["MET_ID"].to_list()
print(f"{twb.height:,} participants with methylation data: "
      f"{len(negative_ids):,} IR-, {len(positive_ids):,} IR+")

1,199 participants with methylation data: 953 IR-, 246 IR+


## Assembling the beta matrix

Probes are restricted to the autosomes — a difference on X or Y would mostly
reflect the sex imbalance between the groups — and to the `cg` prefix, which
excludes the manifest's control and single-nucleotide probes. A probe is then
kept only if **every** participant has a usable reading for it, so that no test
runs on a varying subset. That requirement is what reduces the eligible probes
to the tested set.

In [2]:
probes = autosomal_cg_probes()
matrix = assemble_beta_matrix(probes=probes)
write_parquet(matrix, processed_path("met.parquet"))

print(f"eligible probes {probes.height:,} -> complete in all samples {matrix.height:,}")
print(f"matrix: {matrix.height:,} probes x {matrix.width - 1:,} samples")

2026-09-17 21:30:14 [INFO] src.methylation.extract: Eligible probes: 844316


2026-09-17 21:30:24 [INFO] src.methylation.extract: Probe intersection after 200/1199 samples: 647191


2026-09-17 21:30:34 [INFO] src.methylation.extract: Probe intersection after 400/1199 samples: 567318


2026-09-17 21:30:43 [INFO] src.methylation.extract: Probe intersection after 600/1199 samples: 447834


2026-09-17 21:30:51 [INFO] src.methylation.extract: Probe intersection after 800/1199 samples: 424858


...

2026-09-17 21:31:59 [INFO] src.methylation.extract: Filled 600/1199 samples


2026-09-17 21:32:16 [INFO] src.methylation.extract: Filled 800/1199 samples


2026-09-17 21:32:32 [INFO] src.methylation.extract: Filled 1000/1199 samples


2026-09-17 21:32:52 [INFO] src.methylation.extract: Beta matrix: 332284 probes x 1199 samples


eligible probes 844,316 -> complete in all samples 332,284
matrix: 332,284 probes x 1,199 samples


## Testing every probe

Four tests per probe: a Kolmogorov–Smirnov test of each group against a normal
distribution, the Mann–Whitney U test, and Welch's t-test. With 332,284 probes
the raw p-values are meaningless on their own, so each column is corrected with
the Benjamini–Hochberg procedure; everything downstream uses the corrected
values, and the volcano plot's y axis is a q-value despite its conventional name.

In [3]:
stats = probe_statistics(matrix, negative_ids, positive_ids)
stats = adjust_pvalues(stats)
write_parquet(stats, processed_path("met_p_value.parquet"))

stats.head()

2026-09-17 21:33:03 [INFO] src.methylation.differential: Testing 332284 probes, 953 IR- against 246 IR+


2026-09-17 21:33:19 [INFO] src.methylation.differential: Tested 10000/332284 probes


2026-09-17 21:33:35 [INFO] src.methylation.differential: Tested 20000/332284 probes


2026-09-17 21:33:51 [INFO] src.methylation.differential: Tested 30000/332284 probes


2026-09-17 21:34:07 [INFO] src.methylation.differential: Tested 40000/332284 probes


...

2026-09-17 21:41:29 [INFO] src.methylation.differential: Tested 320000/332284 probes


2026-09-17 21:41:45 [INFO] src.methylation.differential: Tested 330000/332284 probes


2026-09-17 21:41:49 [INFO] src.methylation.differential: Tested 332284/332284 probes


2026-09-17 21:41:49 [INFO] src.methylation.differential: Applied fdr_bh correction across 332284 probes


TargetID,IR-sum,IR-std,IR-n,IR-med,IR+sum,IR+std,IR+n,IR+med,ks_test_p_value_IR-,ks_test_p_value_IR+,u_test_p_value,t_test_p_value,Adj. ks_test_p_value_IR-,Adj. ks_test_p_value_IR+,Adj. u_test_p_value,Adj. t_test_p_value
str,f64,f64,i32,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""cg00000292""",768.615182,0.034104,953,0.798434,198.575742,0.033666,246,0.798811,2.5795e-17,0.00001,0.708767,0.773023,9.0137e-17,0.000043,0.999985,0.999967
"""cg00000596""",644.159104,0.051062,953,0.6759498,166.3046,0.050007,246,0.678191,0.869933,0.401307,0.853107,0.976194,0.882027,0.511101,0.999985,0.999967
"""cg00000721""",855.933317,0.030771,953,0.8915508,220.667121,0.029757,246,0.891606,7.3164e-16,0.000053,0.677066,0.599808,2.3223e-15,0.000182,0.999985,0.999967
"""cg00000734""",79.326316,0.020627,953,0.086318,20.514869,0.022242,246,0.087587,1.7909e-9,0.000279,0.675555,0.921163,3.8675e-9,0.000798,0.999985,0.999967
"""cg00000765""",859.150034,0.029751,953,0.893972,221.205453,0.028995,246,0.892401,1.0965e-27,1.9888e-7,0.116379,0.268032,1.2040e-26,0.000002,0.999985,0.999967


## The volcano plot

Effect size against confidence. The x axis is the log2 ratio of the two group
**medians** — robust to the handful of participants with extreme beta values at
a given probe — and the threshold of 0.2 corresponds to a 14.9% difference in
median methylation. A probe must clear both thresholds to count.

The y axis is the negative log10 of the Benjamini–Hochberg adjusted Mann–Whitney
p-value. It is a q-value, despite the axis label's conventional wording: with
332,284 probes tested, an uncorrected p-value would say almost nothing.

In [4]:
volcano = volcano_frame(stats)
significant = volcano.filter("significant")

print(f"significant: q < {P_VALUE_THRESHOLD}, |log2FC| > {LOG2FC_THRESHOLD}")
print(f"{significant.height} of {volcano.height:,} probes")

plot_volcano(volcano, output_path("volcano_plot_of_DNA_methylation_analysis.png"))
plot_volcano(
    volcano,
    output_path("volcano_plot_of_DNA_methylation_analysis_color_blind.png"),
    significant_color=OKABE_ITO["vermillion"],
    background_color=NEUTRAL_GREY,
    fold_change_color=OKABE_ITO["blue"],
    p_value_color=OKABE_ITO["blue"],
);


2026-09-17 21:41:49 [INFO] src.methylation.differential: Significant probes at q < 0.05 and |log2FC| > 0.2: 22


significant: q < 0.05, |log2FC| > 0.2
22 of 332,284 probes


2026-09-17 21:41:54 [INFO] src.viz.figures: Wrote output/volcano_plot_of_DNA_methylation_analysis.png


2026-09-17 21:41:58 [INFO] src.viz.figures: Wrote output/volcano_plot_of_DNA_methylation_analysis_color_blind.png


## Where the significant probes sit

Each surviving probe is annotated from the EPIC manifest with its gene and its
position relative to the nearest CpG island, then located within the Ensembl
gene track of hg19 to say whether it falls inside a protein-coding sequence.

That last lookup is served from `configs/probe_annotation.csv`, a committed table
rather than a live call to the UCSC REST API, so this notebook runs offline and
the annotation cannot drift under a later re-run.
`src/methylation/annotate.py` carries the query that produced it.

In [5]:
manifest = pl.read_csv(annotation_path(), separator="\t", infer_schema_length=0)

significant_point = (
    significant.select(["TargetID", "neg_log10_p", "log2_fc"])
    .join(manifest, left_on="TargetID", right_on="Probe_ID")
    .sort("neg_log10_p", descending=True, maintain_order=True)
    .select([
        "TargetID", "CHR", "MAPINFO", "neg_log10_p", "log2_fc",
        "UCSC_REFGENE_NAME", "UCSC_REFGENE_GROUP", "RELATION_TO_UCSC_CPG_ISLAND",
    ])
    .with_columns(
        pl.col("UCSC_REFGENE_NAME").fill_null("").str.split(";").list.get(0),
        pl.col("UCSC_REFGENE_GROUP").fill_null("").str.split(";").list.get(0),
        pl.col("RELATION_TO_UCSC_CPG_ISLAND").fill_null(""),
        pl.col("neg_log10_p").round(3),
        pl.col("log2_fc").round(3),
    )
)

ties = significant_point.height - significant_point["neg_log10_p"].n_unique()
print(f"probes sharing a q-value with another (sort-order risk): {ties}")

significant_point = significant_point.join(
    classify_coding_region(load_probe_annotation()), on="TargetID", how="left"
)
significant_point.to_pandas().to_excel(output_path("met-point.xlsx"), index=False)
significant_point

2026-09-17 21:42:00 [INFO] src.methylation.annotate: Region labels: 4 coding, 7 non-coding, 8 unannotated, 3 unlabelled


probes sharing a q-value with another (sort-order risk): 1


TargetID,CHR,MAPINFO,neg_log10_p,log2_fc,UCSC_REFGENE_NAME,UCSC_REFGENE_GROUP,RELATION_TO_UCSC_CPG_ISLAND,coding
str,str,str,f64,f64,str,str,str,str
"""cg00063654""","""3""","""16413077""",4.513,-0.32,"""RFTN1""","""Body""","""""","""coding"""
"""cg00365507""","""16""","""68028125""",3.51,-0.695,"""DPEP2""","""5'UTR""","""S_Shore""","""non_coding"""
"""cg11955727""","""2""","""84105546""",2.812,-0.408,"""""","""""","""""","""non_coding"""
"""cg02987832""","""14""","""35761633""",2.744,0.606,"""PSMA6""","""1stExon""","""Island""","""coding"""
"""cg10661558""","""21""","""15443159""",2.563,-0.222,"""""","""""","""Island""","""non_coding"""
…,…,…,…,…,…,…,…,…
"""cg12385553""","""22""","""17256079""",1.98,0.617,"""""","""""","""""","""no_annotate"""
"""cg01966510""","""15""","""85075427""",1.93,-0.315,"""UBE2Q2P1""","""Body""","""""","""non_coding"""
"""cg07004386""","""14""","""70040391""",1.516,-0.233,"""""","""""","""N_Shore""","""no_annotate"""


## The candidate probe list

The same probes with their full manifest annotation, and the gene symbols the
enrichment analysis takes as input. Probes in intergenic regions carry no gene
symbol and drop out of that list.

In [6]:
candidate = (
    significant_point.select("TargetID")
    .join(manifest, left_on="TargetID", right_on="Probe_ID", maintain_order="left")
    .select(["TargetID", "CHR", "MAPINFO", "^.*UCSC.*$"])
    .rename({"TargetID": "Probe_ID"})
)
candidate.write_csv(output_path("candidate.csv"))

gene_list = sorted(
    set(";".join(significant_point["UCSC_REFGENE_NAME"].unique().to_list()).split(";"))
)
named_genes = [gene for gene in gene_list if gene]
print(f"{len(named_genes)} named genes: {', '.join(named_genes)}")

candidate

13 named genes: B3GNT1, CCNYL1, COL25A1, DPEP2, ERV3-1, FLNB, NHEDC1, PPP1R12B, PSMA6, RFTN1, TFDP1, UBE2Q2P1, UBE2QP1


Probe_ID,CHR,MAPINFO,UCSC_REFGENE_NAME,UCSC_REFGENE_ACCESSION,UCSC_REFGENE_GROUP,UCSC_CPG_ISLANDS_NAME,RELATION_TO_UCSC_CPG_ISLAND
str,str,str,str,str,str,str,str
"""cg00063654""","""3""","""16413077""","""RFTN1""","""NM_015150""","""Body""",null,null
"""cg00365507""","""16""","""68028125""","""DPEP2""","""NM_022355""","""5'UTR""","""chr16:68027074-68027396""","""S_Shore"""
"""cg11955727""","""2""","""84105546""",null,null,null,null,null
"""cg02987832""","""14""","""35761633""","""PSMA6;PSMA6""","""NM_002791;NM_002791""","""1stExon;5'UTR""","""chr14:35761400-35761854""","""Island"""
"""cg10661558""","""21""","""15443159""",null,null,null,"""chr21:15442956-15443160""","""Island"""
…,…,…,…,…,…,…,…
"""cg12385553""","""22""","""17256079""",null,null,null,null,null
"""cg01966510""","""15""","""85075427""","""UBE2Q2P1""","""NR_003661""","""Body""",null,null
"""cg07004386""","""14""","""70040391""",null,null,null,"""chr14:70041213-70041662""","""N_Shore"""


## Pathway enrichment

Are the thirteen genes concentrated in any biological process? Each is tested for
over-representation against ten curated libraries, with Enrichr's own multiple
testing correction within each library.

Read the result with the overlap column in view: most terms are a **single gene**
in a small pathway. Thirteen genes cannot support a strong enrichment claim, and
the combined score rewards exactly that pattern — a small pathway matched by one
gene scores highly. The two terms with an overlap of 2, Focal Adhesion and Myc
Targets V1, are the only ones resting on more than one gene.

Enrichr is a live web service whose libraries are versioned by year and change
over time, so this is the one step of the pipeline that can move without any code
changing. The result table is written to `ORA_result.xlsx` so the figure can be
rebuilt without querying again, and so that what the figure shows stays pinned to
a specific query.

In [7]:
from src.methylation.enrichment import over_representation
from src.viz.figures import plot_enrichment_bars

enrichment = over_representation(named_genes)
enrichment.to_excel(output_path("ORA_result.xlsx"), index=False)

plot_enrichment_bars(enrichment, output_path("ORA.png"))
plot_enrichment_bars(
    enrichment,
    output_path("ORA_color_blind.png"),
    colors=COLOR_BLIND_CATEGORICAL,
)

enrichment[["Gene_set", "Term", "Overlap", "Adjusted P-value", "Combined Score", "Genes"]]

2026-09-17 21:42:24 [INFO] src.methylation.enrichment: Enrichment: 19 terms below adjusted p 0.05, from 13 genes across 10 libraries


2026-09-17 21:42:24 [INFO] src.viz.figures: Wrote output/ORA.png


2026-09-17 21:42:25 [INFO] src.viz.figures: Wrote output/ORA_color_blind.png


,Gene_set,Term,Overlap,Adjusted P-value,Combined Score,Genes
0,HumanCyc_2016,leukotriene biosynthesis Homo sapiens PWY66-375,1/7,0.008420,1497.032601,DPEP2
1,BioCarta_2016,Cyclin E Destruction Pathway Homo sapiens h fb...,1/9,0.025085,1070.484451,TFDP1
2,BioCarta_2016,E2F1 Destruction Pathway Homo sapiens h skp2e2...,1/10,0.025085,932.059746,TFDP1
3,WikiPathways_2024_Human,NPHP1 Deletion Syndrome WP5399,1/11,0.047711,822.994870,FLNB
4,BioCarta_2016,Regulation of p27 Phosphorylation during Cell ...,1/13,0.025085,662.670525,TFDP1
5,HumanCyc_2016,terminal O-glycans residues modification Homo ...,1/13,0.008420,662.670525,B3GNT1
6,WikiPathways_2024_Human,Leukotriene Metabolic Pathway WP5171,1/13,0.047711,662.670525,DPEP2
7,BioCarta_2016,PKC-catalyzed phosphorylation of inhibitory ph...,1/21,0.025085,357.743930,PPP1R12B
8,BioCarta_2016,Cyclins and Cell Cycle Regulation Homo sapiens...,1/23,0.025085,318.354733,TFDP1
9,BioCarta_2016,CCR3 signaling in Eosinophils Homo sapiens h C...,1/23,0.025085,318.354733,PPP1R12B
